In [3]:
import websocket
import json
import time
import uuid

# ── 配置 ──────────────────────────────────────────────────────
TOKEN = "b2dacf6e5ad964021e5c0cbc2788b82e0d7f9ad2a3357bb3"
WS_URL = f"ws://127.0.0.1:18789?token={TOKEN}"

# ── 连接 ──────────────────────────────────────────────────────
ws = websocket.WebSocket()
ws.connect(WS_URL)
print("✅ 已连接 Gateway")

# ── 1. 等待 connect.challenge ─────────────────────────────────
raw = ws.recv()
msg = json.loads(raw)
print(f"← RECV [{msg.get('event', msg.get('type'))}]:", json.dumps(msg, ensure_ascii=False, indent=2))

# ── 2. 发送 connect 握手 ──────────────────────────────────────
connect_req = {
    "type": "req",
    "id": f"py-001",
    "method": "connect",
    "params": {
        "minProtocol": 3,
        "maxProtocol": 10,
        "client": {
            "id": "cli",
            "version": "1.2.3",
            "platform": "python",
            "mode": "cli"
        },
        "role": "operator",
        "scopes": ["operator.read", "operator.write"],
        "auth": {"token": TOKEN},
    }
}
ws.send(json.dumps(connect_req))
print("→ SEND [connect]")

# ── 3. 等待握手响应 ───────────────────────────────────────────
raw = ws.recv()
res = json.loads(raw)
print("← RECV [connect.response]:", json.dumps(res, ensure_ascii=False, indent=2))
if res.get("ok"):
    print("✅ 握手成功，可以发送消息")
else:
    print("❌ 握手失败:", res.get("error"))

# 4. 向session发送消息
chat_req = {
  "type": "req",
  "id": "py-001",
  "method": "chat.send",
  "params": {
    "sessionKey": "agent:main:main",
    "message": "Hello, world!",
    "idempotencyKey": uuid.uuid4().hex
  }
}
ws.send(json.dumps(chat_req))
print("→ SEND [chat.send]")

raw = ws.recv()
res = json.loads(raw)
print("← RECV [chat.send.response]:", json.dumps(res, ensure_ascii=False, indent=2))

raw = ws.recv()
res = json.loads(raw)
print("← RECV [chat.send.response2]:", json.dumps(res, ensure_ascii=False, indent=2))

input()
ws.close()

✅ 已连接 Gateway
← RECV [connect.challenge]: {
  "type": "event",
  "event": "connect.challenge",
  "payload": {
    "nonce": "f7dc4b4b-801c-46bf-87bd-7c3b05556997",
    "ts": 1777378495414
  }
}
→ SEND [connect]
← RECV [connect.response]: {
  "type": "res",
  "id": "py-001",
  "ok": true,
  "payload": {
    "type": "hello-ok",
    "protocol": 3,
    "server": {
      "version": "2026.4.24",
      "connId": "5c312fad-17b9-4fcf-81f3-ee965e7afdb0"
    },
    "features": {
      "methods": [
        "health",
        "diagnostics.stability",
        "doctor.memory.status",
        "doctor.memory.dreamDiary",
        "doctor.memory.backfillDreamDiary",
        "doctor.memory.resetDreamDiary",
        "doctor.memory.resetGroundedShortTerm",
        "doctor.memory.repairDreamingArtifacts",
        "doctor.memory.dedupeDreamDiary",
        "logs.tail",
        "channels.status",
        "channels.start",
        "channels.logout",
        "status",
        "usage.status",
        "usage.cost",
 